# RunPod GPU
Keep **this notebook on your local `.venv` kernel**. Only your training notebook uses the remote Jupyter URL.

## 0 · GPUs and prices (optional, local kernel)
Available GPUs, cheapest first. Copy a name into `GPU` in step 1. Memory needs depend on the model and batch size.

In [ ]:
import json, subprocess

gpus = json.loads(subprocess.check_output(["runpodctl", "gpu", "list"]))
for g in sorted(gpus, key=lambda g: g.get("securePricePerHr") or 99):
    if g.get("securePricePerHr") and "NVIDIA" in g["gpuId"] and "MIG" not in g["gpuId"]:
        print(f'${g["securePricePerHr"]}/hr  {g["memoryInGb"]} GB  {g["gpuId"]}')

## 1 · Start a GPU (local kernel)
Copies code and data; skips saved checkpoints. Installs pinned packages on Linux and prints only the ready Jupyter URL. Setup failure deletes this attempt’s pod.
**Charges start now.** Run once; use step 0 for prices.

In [ ]:
import json, secrets, shlex, subprocess

GPU = "NVIDIA A40"
IMAGE = "runpod/pytorch:1.0.3-cu1281-torch291-ubuntu2404"  # matches requirements.txt

def run(command):
    result = subprocess.run(command, capture_output=True, text=True, timeout=900)
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout

repo = run(["git", "rev-parse", "--show-toplevel"]).strip()
pods = json.loads(run(["runpodctl", "pod", "list", "--all"]))
if any(p["name"].startswith("learn-gpu") for p in pods):
    raise RuntimeError("A learning pod already exists. Use it or delete it first.")

name = "learn-gpu-" + secrets.token_hex(3)
pw = secrets.token_urlsafe(24)
target = "/workspace/LLM-fine-tuning"

try:
    pod = json.loads(run([
        "runpodctl", "pod", "create", "--name", name, "--image", IMAGE,
        "--gpu-id", GPU, "--min-cuda-version", "12.8", "--ports", "8888/http,22/tcp",
        "--container-disk-in-gb", "30", "--volume-in-gb", "50",
        "--env", json.dumps({"JUPYTER_PASSWORD": pw}), "--wait"]))
    ssh = shlex.split(pod["ssh"]["ssh_command"]) + [
        "-o", "StrictHostKeyChecking=accept-new", "-o", "BatchMode=yes",
        "-o", "ConnectTimeout=15", "-o", "ServerAliveInterval=15", "-o", "ServerAliveCountMax=3"]

    # Copy code and data; skip saved checkpoints, local environments, secrets and caches.
    skip = ".git .venv __pycache__ .ipynb_checkpoints .DS_Store ._* *.pyc .env .env.* .vscode .idea *.pth *.pt *.safetensors".split()
    pack = ["tar", "--no-mac-metadata", "--no-xattrs", *["--exclude=" + s for s in skip],
            "-C", repo, "-cf", "-", "."]
    unpack = ssh + [f"mkdir -p {target} && tar --no-same-owner -xf - -C {target}"]
    run(["bash", "-o", "pipefail", "-c", shlex.join(pack) + " | " + shlex.join(unpack)])

    # Install into the pod’s Jupyter Python; confirm CUDA works.
    run(ssh + [f"python -m pip install --break-system-packages --no-cache-dir -r {target}/requirements.txt && "
               "python -c 'import torch; torch.zeros(1).cuda()'"])
    url = f"https://{pod['id']}-8888.proxy.runpod.net/?token={pw}"
    run(["curl", "-fsS", "--retry", "10", "--retry-delay", "2", "--retry-connrefused",
         "--max-time", "10", url.replace("/?", "/api?")])
except BaseException as error:
    # Delete this attempt's pod if setup fails or is interrupted.
    try:
        for p in json.loads(run(["runpodctl", "pod", "list", "--all"])):
            if p["name"] == name:
                run(["runpodctl", "pod", "delete", p["id"]])
        if any(p["name"] == name for p in json.loads(run(["runpodctl", "pod", "list", "--all"]))):
            raise RuntimeError("Pod still exists.")
    except Exception as cleanup_error:
        raise RuntimeError(f"Cleanup failed: delete {name} in the Runpod console. {cleanup_error}") from error
    raise

print(url)


### Upload one checkpoint (optional, local kernel)
After Start, uncomment the last line below and enter your checkpoint path.
Saves to `Training/runs/instruction/002_finetuned/` on the pod.


In [ ]:
from pathlib import Path


def upload_checkpoint(path):
    path = Path(path).expanduser().resolve(strict=True)
    folder = f"{target}/Training/runs/instruction/002_finetuned"
    run(ssh + [f"mkdir -p {shlex.quote(folder)}"])
    subprocess.run([
        "scp", "-i", pod["ssh"]["ssh_key"]["path"], "-P", str(pod["ssh"]["port"]),
        "-o", "BatchMode=yes", str(path), f"root@{pod['ssh']['ip']}:{folder}/"
    ], check=True)

# upload_checkpoint("/full/path/to/checkpoint.pth")

# Upload the latest checkpoint.
# upload_checkpoint(max((Path(repo) / "Training/runs/instruction/002_finetuned").glob("checkpoint_*.pth")))

## 2 · Connect your training notebook
Open `Training/03_instruction.ipynb`: **Select Kernel → Select Another Kernel → Existing Jupyter Server** → paste the URL → choose **Python 3 (ipykernel)**. Keep `setup.ipynb` local.

## Terminal (optional)
In this local kernel, run `print(pod["ssh"]["ssh_command"])`, then paste that command into your terminal. `exit` returns to your Mac.

## 3 · Train
In your training notebook, run its Runpod `%cd` cell, then your model/data/training cells. Its device should print `cuda`.

## Files & scripts
Project: `/workspace/LLM-fine-tuning`. For training notebooks, first run `%cd /workspace/LLM-fine-tuning/Training` (skip locally).

- Cell edits run immediately. Changed local scripts or data need copying again.
- Code/data stay on pod disk; `.to(device)` moves the model and tensor batches to GPU memory.
- To retrieve results: GPU terminal `runpodctl send model.pth` → local terminal `runpodctl receive <code>`. Save results before deleting.

### Download the latest checkpoint only
After Save, run in the **remote training notebook**:
```python
!runpodctl send {max(folder.glob('checkpoint_*.pth'))}
```
Run in your **local terminal**, inside `Training/runs/instruction/002_finetuned/`:
```sh
runpodctl receive <code>
```
Use the code printed by `send`. Wait for completion before deleting the pod.


## 4 · Delete the pod (local kernel)
Return here when finished or after a training error. Download results first; this deletes the pod and its disk. If setup failed, skip to step 5.

In [ ]:
!runpodctl pod delete {pod['id']}

## 5 · Confirm (local kernel)
`[]` means no pods remain. If a pod is listed, delete its ID with `!runpodctl pod delete <id>`; replace `<id>` with the listed ID. This also works after restarting your local kernel.

In [ ]:
!runpodctl pod list --all